# Plano 1 — Modelagem preditiva e métricas (5.4 + 6.1)

Walk-forward por posição: Baseline, Ridge, XGBoost, LightGBM, Random Forest e ensemble ponderado.

Para cada rodada de teste `t`, treina com `4 <= rodada_id < t` e testa só em `t`.

- Padrão (primeiro turno): rodadas 5–18, outputs sem sufixo.
- Segundo turno: `python models/run_train_eval.py --start 19 --end 34 --tag _segundo_turno --seed-results models/outputs/resultados_walkforward.csv` (outputs com sufixo `_segundo_turno`; o seed só alimenta os pesos do ensemble na rodada 19).

**Fora de escopo:** PLI, orçamento, capitão, escalações.

## 0. Imports e configuração

In [ ]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Imports OK")

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "database" / "players.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/database/players.csv")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.eligibility import STATUS_CANDIDATOS
from models.run_config import load_run_config

RUN_CONFIG = load_run_config(os.environ)
PATH_FEATURES = PROJECT_ROOT / "data" / "database" / "df_features.csv"
PATH_OUT = PROJECT_ROOT / "models" / "outputs"
PATH_OUT.mkdir(parents=True, exist_ok=True)

POS_MAP = {
    1: "Goleiro",
    2: "Lateral",
    3: "Zagueiro",
    4: "Meia",
    5: "Atacante",
    6: "Técnico",
}

MIN_TRAIN_ROUND = 4
MIN_TEST_ROUND = RUN_CONFIG.test_round_start
N_ROUNDS = RUN_CONFIG.test_round_end
OUTPUT_TAG = RUN_CONFIG.output_tag
N_TRIALS = 20
ENSEMBLE_K = 3
ENSEMBLE_MODELS = ["ridge", "xgboost", "lightgbm", "rf"]
MODELOS = ["baseline", "ridge", "xgboost", "lightgbm", "rf", "ensemble"]

COLS_DROP = [
    "rodada_id",
    "atleta_id",
    "apelido",
    "posicao_id",
    "clube_id",
    "adversario_id",
    "pontos_num",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PATH_FEATURES:", PATH_FEATURES)
print("PATH_OUT:", PATH_OUT)
print("RUN_CONFIG:", RUN_CONFIG)

## 1. Carrega dados e define FEATURE_COLS

H2H **não** é imputado globalmente aqui (evita look-ahead). A imputação é fold-safe no walk-forward.

In [ ]:
df = pd.read_csv(PATH_FEATURES)

FEATURE_COLS = [c for c in df.columns if c not in COLS_DROP]
assert "pontos_num" in df.columns, "Target pontos_num missing"
assert "h2h_pts_avg" in FEATURE_COLS, "h2h_pts_avg must be a feature"
assert "status_id" in df.columns, "status_id required for candidate filter"

print(f"df_features: {df.shape}")
print(f"Rodadas: {sorted(df['rodada_id'].unique())}")
print(f"Features: {len(FEATURE_COLS)}")
print(
    "NaNs (pre-imputation):\n",
    df[FEATURE_COLS].isnull().sum()[df[FEATURE_COLS].isnull().sum() > 0].to_string(),
)
df.head(3)

## 2. Trainers, métricas e imputação fold-safe

In [ ]:
def impute_h2h(train: pd.DataFrame, test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Impute h2h_pts_avg using only training-fold statistics."""
    train = train.copy()
    test = test.copy()

    athlete_means = train.groupby("atleta_id")["h2h_pts_avg"].mean()
    global_mean = float(train["h2h_pts_avg"].mean())
    if np.isnan(global_mean):
        global_mean = 0.0

    train["h2h_pts_avg"] = train["h2h_pts_avg"].fillna(
        train["atleta_id"].map(athlete_means)
    )
    train["h2h_pts_avg"] = train["h2h_pts_avg"].fillna(global_mean)

    test["h2h_pts_avg"] = test["h2h_pts_avg"].fillna(test["atleta_id"].map(athlete_means))
    test["h2h_pts_avg"] = test["h2h_pts_avg"].fillna(global_mean)
    return train, test


def baseline_predict(df_test: pd.DataFrame) -> np.ndarray:
    return (
        df_test["pts_avg_5r"]
        .fillna(df_test["pts_avg_3r"])
        .fillna(0.0)
        .to_numpy(dtype=float)
    )



def optuna_mae(model, X_train, y_train, tscv):
    """Temporal validation MAE for Optuna (holdout preferred over full CV)."""
    n = len(y_train)
    if n >= 30:
        split = max(int(n * 0.8), n - max(5, n // 5))
        model.fit(X_train[:split], y_train[:split])
        return float(mean_absolute_error(y_train[split:], model.predict(X_train[split:])))
    if tscv is None:
        model.fit(X_train, y_train)
        return float(mean_absolute_error(y_train, model.predict(X_train)))
    scores = -cross_val_score(
        model, X_train, y_train, cv=tscv, scoring="neg_mean_absolute_error"
    )
    return float(scores.mean())


def make_tscv(n_samples: int):
    n_splits = min(3, max(2, n_samples // 5))
    if n_samples <= n_splits + 1:
        return None
    return TimeSeriesSplit(n_splits=n_splits)


def treina_ridge(X_train, y_train):
    pipe = Pipeline([("scaler", StandardScaler()), ("model", Ridge())])
    tscv = make_tscv(len(y_train))
    alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
    best_alpha, best_score = 1.0, np.inf
    if tscv is None:
        for alpha in alphas:
            pipe.set_params(model__alpha=alpha)
            pipe.fit(X_train, y_train)
            score = mean_absolute_error(y_train, pipe.predict(X_train))
            if score < best_score:
                best_score, best_alpha = score, alpha
    else:
        for alpha in alphas:
            pipe.set_params(model__alpha=alpha)
            scores = -cross_val_score(
                pipe, X_train, y_train, cv=tscv, scoring="neg_mean_absolute_error"
            )
            if scores.mean() < best_score:
                best_score, best_alpha = scores.mean(), alpha
    pipe.set_params(model__alpha=best_alpha)
    pipe.fit(X_train, y_train)
    return pipe


def treina_xgboost(X_train, y_train, n_trials: int = N_TRIALS):
    tscv = make_tscv(len(y_train))

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 400),
            "max_depth": trial.suggest_int("max_depth", 2, 7),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "random_state": 42,
            "n_jobs": -1,
        }
        model = xgb.XGBRegressor(**params, verbosity=0)
        return optuna_mae(model, X_train, y_train, tscv)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = xgb.XGBRegressor(
        **study.best_params, verbosity=0, random_state=42, n_jobs=-1
    )
    best.fit(X_train, y_train)
    return best


def treina_lightgbm(X_train, y_train, n_trials: int = N_TRIALS):
    tscv = make_tscv(len(y_train))

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 400),
            "max_depth": trial.suggest_int("max_depth", 2, 7),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 10, 100),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "random_state": 42,
            "n_jobs": -1,
            "verbose": -1,
        }
        model = lgb.LGBMRegressor(**params)
        return optuna_mae(model, X_train, y_train, tscv)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = lgb.LGBMRegressor(
        **study.best_params, random_state=42, n_jobs=-1, verbose=-1
    )
    best.fit(X_train, y_train)
    return best


def treina_rf(X_train, y_train):
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def calcula_metricas(y_true, y_pred, nome_modelo, rodada, posicao):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan
    rho, pval = spearmanr(y_true, y_pred)
    n = len(y_true)
    if n >= 3:
        top3_real = set(np.argsort(y_true)[-3:])
        top3_pred = set(np.argsort(y_pred)[-3:])
        top3_prec = len(top3_real & top3_pred) / 3.0
    else:
        top3_prec = np.nan
    return {
        "modelo": nome_modelo,
        "rodada": rodada,
        "posicao": posicao,
        "n": n,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "spearman": float(rho) if rho == rho else np.nan,
        "spearman_p": float(pval) if pval == pval else np.nan,
        "top3_precision": top3_prec,
    }


def ensemble_weights(resultados: list[dict], posicao: str, k: int = ENSEMBLE_K) -> dict[str, float]:
    """Inverse-MAE weights from the last k rounds for this position."""
    hist = [
        r
        for r in resultados
        if r["posicao"] == posicao and r["modelo"] in ENSEMBLE_MODELS
    ]
    if not hist:
        return {m: 1.0 / len(ENSEMBLE_MODELS) for m in ENSEMBLE_MODELS}

    last_rounds = sorted({r["rodada"] for r in hist})[-k:]
    mae_by_model = {}
    for model in ENSEMBLE_MODELS:
        maes = [
            r["mae"]
            for r in hist
            if r["modelo"] == model and r["rodada"] in last_rounds and np.isfinite(r["mae"])
        ]
        if maes:
            mae_by_model[model] = float(np.mean(maes))

    if not mae_by_model:
        return {m: 1.0 / len(ENSEMBLE_MODELS) for m in ENSEMBLE_MODELS}

    inv = {m: 1.0 / (mae + 1e-6) for m, mae in mae_by_model.items()}
    total = sum(inv.values())
    return {m: inv.get(m, 0.0) / total for m in ENSEMBLE_MODELS}


def ensemble_predict(preds: dict[str, np.ndarray], weights: dict[str, float]) -> np.ndarray:
    out = None
    weight_sum = 0.0
    for name, arr in preds.items():
        w = weights.get(name, 0.0)
        if w <= 0:
            continue
        out = arr.astype(float) * w if out is None else out + arr.astype(float) * w
        weight_sum += w
    if out is None or weight_sum <= 0:
        stacked = np.vstack([preds[m] for m in ENSEMBLE_MODELS if m in preds])
        return stacked.mean(axis=0)
    return out / weight_sum


print("Trainers OK")


## 3. Walk-forward (só predição / métricas)

In [ ]:
resultados_modelos: list[dict] = []
if RUN_CONFIG.seed_results_path is not None:
    df_seed = pd.read_csv(RUN_CONFIG.seed_results_path)
    df_seed = df_seed[df_seed["rodada"] < MIN_TEST_ROUND]
    resultados_modelos.extend(df_seed.to_dict("records"))
    print(
        f"Seed do ensemble: {len(df_seed)} linhas, rodadas "
        f"{int(df_seed['rodada'].min())}–{int(df_seed['rodada'].max())}"
    )
N_SEED_RESULTS = len(resultados_modelos)
feature_importances: dict[tuple, pd.Series] = {}
predicoes_rows: list[dict] = []

rodadas_teste = RUN_CONFIG.test_rounds
print(f"Rodadas de teste: {rodadas_teste}")

for t in rodadas_teste:
    print(f"\n{'=' * 60}\nRODADA {t}\n{'=' * 60}")

    df_treino = df[(df["rodada_id"] < t) & (df["rodada_id"] >= MIN_TRAIN_ROUND)].copy()
    df_teste = df[df["rodada_id"] == t].copy()
    if "status_id" in df_teste.columns:
        df_teste = df_teste[df_teste["status_id"].isin(STATUS_CANDIDATOS)].copy()

    assert (df_treino["rodada_id"] < t).all(), "Look-ahead in training fold"
    assert (df_teste["rodada_id"] == t).all(), "Test fold must be exactly round t"

    df_treino, df_teste = impute_h2h(df_treino, df_teste)

    for pos_id, pos_nome in POS_MAP.items():
        df_tr_pos = df_treino[df_treino["posicao_id"] == pos_id].copy()
        df_te_pos = df_teste[df_teste["posicao_id"] == pos_id].copy()

        if len(df_te_pos) == 0:
            print(f"  [{pos_nome}] sem candidatos — skip")
            continue

        y_true = df_te_pos["pontos_num"].to_numpy(dtype=float)
        preds_pos: dict[str, np.ndarray] = {}

        if len(df_tr_pos) < 10:
            base = baseline_predict(df_te_pos)
            for nome in MODELOS:
                preds_pos[nome] = base
            print(f"  [{pos_nome}] treino insuficiente — baseline fallback")
        else:
            X_train = df_tr_pos[FEATURE_COLS].fillna(0).to_numpy(dtype=float)
            y_train = df_tr_pos["pontos_num"].to_numpy(dtype=float)
            X_test = df_te_pos[FEATURE_COLS].fillna(0).to_numpy(dtype=float)

            preds_pos["baseline"] = baseline_predict(df_te_pos)

            try:
                m_ridge = treina_ridge(X_train, y_train)
                preds_pos["ridge"] = m_ridge.predict(X_test)
            except Exception as exc:
                preds_pos["ridge"] = preds_pos["baseline"]
                print(f"  [ERRO Ridge {pos_nome}]: {exc}")

            try:
                m_xgb = treina_xgboost(X_train, y_train, n_trials=N_TRIALS)
                preds_pos["xgboost"] = m_xgb.predict(X_test)
                feature_importances[(t, pos_nome, "xgboost")] = pd.Series(
                    m_xgb.feature_importances_, index=FEATURE_COLS
                )
            except Exception as exc:
                preds_pos["xgboost"] = preds_pos["baseline"]
                print(f"  [ERRO XGBoost {pos_nome}]: {exc}")

            try:
                m_lgb = treina_lightgbm(X_train, y_train, n_trials=N_TRIALS)
                preds_pos["lightgbm"] = m_lgb.predict(X_test)
            except Exception as exc:
                preds_pos["lightgbm"] = preds_pos["baseline"]
                print(f"  [ERRO LightGBM {pos_nome}]: {exc}")

            try:
                m_rf = treina_rf(X_train, y_train)
                preds_pos["rf"] = m_rf.predict(X_test)
                feature_importances[(t, pos_nome, "rf")] = pd.Series(
                    m_rf.feature_importances_, index=FEATURE_COLS
                )
            except Exception as exc:
                preds_pos["rf"] = preds_pos["baseline"]
                print(f"  [ERRO RF {pos_nome}]: {exc}")

            weights = ensemble_weights(resultados_modelos, pos_nome, k=ENSEMBLE_K)
            preds_pos["ensemble"] = ensemble_predict(
                {m: preds_pos[m] for m in ENSEMBLE_MODELS}, weights
            )

        for nome, y_pred in preds_pos.items():
            resultados_modelos.append(
                calcula_metricas(y_true, y_pred, nome, t, pos_nome)
            )

        for i, (_, row) in enumerate(df_te_pos.iterrows()):
            rec = {
                "rodada": t,
                "atleta_id": int(row["atleta_id"]),
                "apelido": row.get("apelido", ""),
                "posicao_id": int(pos_id),
                "posicao": pos_nome,
                "y_true": float(y_true[i]),
            }
            for nome, y_pred in preds_pos.items():
                rec[f"y_hat_{nome}"] = float(y_pred[i])
            predicoes_rows.append(rec)

        print(
            f"  [{pos_nome}] n_test={len(df_te_pos)} | "
            f"MAE xgb={mean_absolute_error(y_true, preds_pos['xgboost']):.3f}"
        )

print("\nWalk-forward concluído")

## 4. Salva CSVs

In [ ]:
df_resultados = pd.DataFrame(resultados_modelos[N_SEED_RESULTS:])
assert df_resultados["rodada"].between(MIN_TEST_ROUND, N_ROUNDS).all()
df_predicoes = pd.DataFrame(predicoes_rows)

fi_records = []
for (rodada, posicao, modelo), fi_series in feature_importances.items():
    for feat, imp in fi_series.items():
        fi_records.append(
            {
                "rodada": rodada,
                "posicao": posicao,
                "modelo": modelo,
                "feature": feat,
                "importance": float(imp),
            }
        )
df_fi = pd.DataFrame(fi_records)

tab_modelos = (
    df_resultados.groupby("modelo")
    .agg(
        RMSE=("rmse", "mean"),
        MAE=("mae", "mean"),
        R2=("r2", "mean"),
        Spearman=("spearman", "mean"),
        Top3Prec=("top3_precision", "mean"),
        N=("n", "sum"),
    )
    .round(3)
    .sort_values("MAE")
)

tab_por_posicao = (
    df_resultados.groupby(["modelo", "posicao"])
    .agg(
        RMSE=("rmse", "mean"),
        MAE=("mae", "mean"),
        R2=("r2", "mean"),
        Spearman=("spearman", "mean"),
        Top3Prec=("top3_precision", "mean"),
        N=("n", "sum"),
    )
    .round(3)
    .reset_index()
)

out_csv = RUN_CONFIG.output_name
df_resultados.to_csv(PATH_OUT / out_csv("resultados_walkforward"), index=False)
df_fi.to_csv(PATH_OUT / out_csv("feature_importances"), index=False)
df_predicoes.to_csv(PATH_OUT / out_csv("predicoes_walkforward"), index=False)
tab_modelos.to_csv(PATH_OUT / out_csv("tabela_6_1"))
tab_por_posicao.to_csv(PATH_OUT / out_csv("tabela_6_1_por_posicao"), index=False)

print("Outputs salvos em", PATH_OUT)
print(tab_modelos)

## 5. Figuras 6.1

In [ ]:
pivot_rmse = df_resultados.pivot_table(
    index="posicao", columns="modelo", values="rmse", aggfunc="mean"
).round(3)
pivot_spe = df_resultados.pivot_table(
    index="posicao", columns="modelo", values="spearman", aggfunc="mean"
).round(3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ROUNDS_LABEL = f"rodadas {MIN_TEST_ROUND}–{N_ROUNDS}"
fig.suptitle(
    f"Comparação de Modelos por Posição ({ROUNDS_LABEL})", fontsize=13, fontweight="bold"
)

pivot_rmse.plot(kind="bar", ax=axes[0], colormap="tab10", edgecolor="white")
axes[0].set_title("RMSE por Posição (menor = melhor)", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("RMSE (pts)")
axes[0].legend(title="Modelo", bbox_to_anchor=(1, 1))
axes[0].tick_params(axis="x", rotation=30)

pivot_spe.plot(kind="bar", ax=axes[1], colormap="tab10", edgecolor="white")
axes[1].set_title("Spearman ρ por Posição (maior = melhor)", fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Spearman ρ")
axes[1].legend(title="Modelo", bbox_to_anchor=(1, 1))
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
fig_name = RUN_CONFIG.output_name("fig_comparacao_modelos", ".png")
plt.savefig(PATH_OUT / fig_name, dpi=150, bbox_inches="tight")
plt.close()
print(fig_name, "OK")

In [ ]:
df_fi_xgb = df_fi[df_fi["modelo"] == "xgboost"]
posicoes_plot = list(POS_MAP.values())

fig, axes = plt.subplots(2, 3, figsize=(16, 12))
fig.suptitle(
    f"Feature Importance — XGBoost por Posição (Top 15, {ROUNDS_LABEL})",
    fontsize=13,
    fontweight="bold",
)

for ax, pos in zip(axes.flat, posicoes_plot):
    fi_pos = (
        df_fi_xgb[df_fi_xgb["posicao"] == pos]
        .groupby("feature")["importance"]
        .mean()
        .sort_values(ascending=False)
        .head(15)
    )
    if fi_pos.empty:
        ax.set_visible(False)
        continue
    fi_pos[::-1].plot(kind="barh", ax=ax, color="#003366", alpha=0.8)
    ax.set_title(pos, fontweight="bold")
    ax.set_xlabel("Importância média")
    ax.grid(alpha=0.3)

plt.tight_layout()
fig_name = RUN_CONFIG.output_name("fig_feature_importance", ".png")
plt.savefig(PATH_OUT / fig_name, dpi=150, bbox_inches="tight")
plt.close()
print(fig_name, "OK")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    f"Pontos Previstos vs. Reais por Posição (XGBoost, walk-forward, {ROUNDS_LABEL})",
    fontsize=13,
    fontweight="bold",
)

for ax, (pos_id, pos_nome) in zip(axes.flat, POS_MAP.items()):
    sub = df_predicoes[df_predicoes["posicao"] == pos_nome]
    if sub.empty:
        ax.set_visible(False)
        continue
    ax.scatter(
        sub["y_hat_xgboost"],
        sub["y_true"],
        alpha=0.35,
        s=18,
        color="#003366",
    )
    lim = max(sub["y_hat_xgboost"].max(), sub["y_true"].max(), 1.0) * 1.05
    ax.plot([0, lim], [0, lim], "r--", alpha=0.6, linewidth=1)
    rho = sub[["y_hat_xgboost", "y_true"]].corr(method="spearman").iloc[0, 1]
    mae = (sub["y_hat_xgboost"] - sub["y_true"]).abs().mean()
    ax.set_title(f"{pos_nome}  |  ρ={rho:.2f}, MAE={mae:.2f}", fontweight="bold")
    ax.set_xlabel("Previsto (pts)")
    ax.set_ylabel("Real (pts)")
    ax.grid(alpha=0.25)

plt.tight_layout()
fig_name = RUN_CONFIG.output_name("fig_previsto_vs_real", ".png")
plt.savefig(PATH_OUT / fig_name, dpi=150, bbox_inches="tight")
plt.close()
print(fig_name, "OK")
print("\n=== Tabela 6.1 ===")
print(tab_modelos.to_string())

In [ ]:
metricas_rodada = (
    df_resultados.groupby(["rodada", "modelo"])[["rmse", "spearman"]]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(
    f"Métricas por Rodada (média das posições, {ROUNDS_LABEL})",
    fontsize=13,
    fontweight="bold",
)
for ax, metric, label in [
    (axes[0], "rmse", "RMSE (pts) — menor = melhor"),
    (axes[1], "spearman", "Spearman ρ — maior = melhor"),
]:
    pivot = metricas_rodada.pivot(index="rodada", columns="modelo", values=metric)
    pivot.plot(ax=ax, marker="o", linewidth=1.5, colormap="tab10")
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("Rodada")
    ax.set_xticks(rodadas_teste)
    ax.grid(alpha=0.3)
    ax.legend(title="Modelo", fontsize=8)

plt.tight_layout()
fig_name = RUN_CONFIG.output_name("fig_metricas_por_rodada", ".png")
plt.savefig(PATH_OUT / fig_name, dpi=150, bbox_inches="tight")
plt.close()
print(fig_name, "OK")